# FreshRoute — Plan v3.1 Pipeline on Colab
Food Bank Distribution Optimizer (India): 5-key deterministic cleaning -> Train-only features & `10d/7d/4d` chronological pairing -> Track 1A (Glut Onset) & Track 1B (`reg:absoluteerror` Arrival Residuals + Surge Ranking) -> 5 Automated Anti-Leakage Checks.

**Run top to bottom.** Upload `apmc_arrivals_prices.csv` (81 MB, re-download steps in `data/SOURCES.md`) when prompted.

In [ ]:
# 0 — Clone repo + install dependencies
import os
if not os.path.isdir('/content/freshroute'):
    !git clone https://github.com/ParasRana1729/freshroute /content/freshroute
%cd /content/freshroute
!pip install -q -r requirements.txt
print('repo ready:', os.getcwd())

In [ ]:
# 1 — Provide raw APMC data (git-ignored)
import os
RAW = 'data/raw/apmc_arrivals_prices.csv'
if not os.path.exists(RAW):
    print('Upload apmc_arrivals_prices.csv when prompted (81 MB)')
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    os.makedirs('data/raw', exist_ok=True)
    os.replace(name, RAW)
import pandas as pd
df0 = pd.read_csv(RAW, low_memory=False)
print('raw shape:', df0.shape)
df0.head(3)

In [ ]:
# 2 — Phase 0: 5-key deterministic row cleaning (zero cross-split statistics)
!python scripts/clean_dataset.py

In [ ]:
# 3 — Verify cleaning outputs
import json, pandas as pd
rep = json.load(open('data/processed/cleaning_report.json'))
print('cleaned:', rep['cleaned_shape'], '| encoded:', rep['encoded_shape'], '| entities:', rep['entity_count'])
print('dedup removed:', rep['dedup_duplicates_removed'])
clean = pd.read_csv('data/processed/freshroute_foodbank_cleaned.csv', nrows=5)
clean[['date','state_name','market_center_name','commodity_name','variety','grade','arrival_tonnes','price_per_kg']]

In [ ]:
# 4 — Phase 0-1: Train-only IQR, 4-level shrinkage priors & 10d/7d/4d chronological pairing
!python scripts/build_features.py

In [ ]:
# 5 — Phase 1: Train Track 1A (Glut Onset) & Track 1B (delta_log_arr + Surge NDCG@K)
!python scripts/train_model.py

In [ ]:
# 6 — Inspect Track 1A & Track 1B Test metrics + top dispatch list
import json, pandas as pd
m = json.load(open('data/processed/model_metrics.json'))
print('Split:', m['split_mode'], '| Test dates:', m['split_stats']['test_target_dates'])
print('--- Track 1A (Glut Onset) Test ---')
display(pd.DataFrame({k: v['test'] for k, v in m['track_1a']['models'].items()}).T)
print('--- Track 1B (delta_log_arr) Test ---')
display(pd.DataFrame({k: v['test'] for k, v in m['track_1b']['models'].items()}).T)
disp = pd.read_csv('data/processed/dispatch_list_demo.csv')
print('dispatch rows:', len(disp))
disp.head(10)

In [ ]:
# 7 — Phase 3: Run 5 automated anti-leakage & shuffled-target collapse tests
!python -m pytest -v tests/test_pipeline_integrity.py tests/test_phase2_pipelines.py